# Seasonal evaluation report

Scores of the AI ensembles over the season in the config, against each truth. Run the stages first:

```bash
uv run scripts/run_season.py configs/summer2026_france.yaml --stage forecast   # GPU
uv run scripts/run_season.py configs/summer2026_france.yaml --stage score      # CPU
```

The notebook only reads the saved per-case statistics (`outputs/season/<name>/scores/`), so it runs in seconds and can be re-run while the season is still being forecast: it reports whatever is scored so far.

All scores are fair estimators (comparable across ensemble sizes) and are computed from summed statistics over the selected init times, never as averages of per-case scores. Lower is better for every score except the spread-skill ratio, which should be 1.

In [ ]:
# Parameters
CONFIG = "configs/summer2026_france.yaml"
OVERRIDES = {}          # e.g. {"period": {"season": "JJA", "year": 2025}}
LEADS_FOR_TABLES = [24, 72, 120]   # hours

In [ ]:
import os, sys
from pathlib import Path

# run from the repo root wherever the notebook lives
root = Path.cwd()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from weather_pipeline.season import config
from weather_pipeline.season.score import load_statistics, season_scores

cfg = config.load(CONFIG, **OVERRIDES)
print(cfg.describe())

stats = {truth: load_statistics(cfg, truth) for truth in cfg.truths}
scored = {truth: {m: int(s.sizes["init_time"]) for m, s in by_model.items()} for truth, by_model in stats.items()}
print("\nscored initialisations:", scored)
stats = {t: s for t, s in stats.items() if s}
assert stats, "Nothing scored yet: run --stage score first"


## Season scores

Each metric is finalised over all scored initialisations. `n_init` differing between models means the forecast stage has not finished one of them yet: compare models only over common dates (next cell).

In [ ]:
def common_inits(by_model):
    sets = [set(s["init_time"].values) for s in by_model.values()]
    return np.array(sorted(set.intersection(*sets)))

metrics = {truth: {m: season_scores(s, common_inits(by_model)) for m, s in by_model.items()}
           for truth, by_model in stats.items()}
for truth, by_model in stats.items():
    print(f"{truth}: {len(common_inits(by_model))} initialisation(s) common to all models")


In [ ]:
hours = lambda da: da["lead_time"].values / np.timedelta64(1, "h")
panels = [("crps", "CRPS (K)"), ("ens_mean_rmse", "ensemble-mean RMSE (K)"),
          ("spread_skill", "spread-skill ratio"), ("scrps", "scaled CRPS"),
          ("energy_score", "energy score (K)"), ("large_error_frac", "fraction of SE > 4 K²")]
for truth, by_model in metrics.items():
    fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True)
    for ax, (key, label) in zip(axes.flat, panels):
        for model, m in by_model.items():
            ax.plot(hours(m[key]), m[key], marker="o", ms=3, label=model)
        if key == "spread_skill":
            ax.axhline(1, color="k", lw=0.8, ls="--")
        ax.set_title(label); ax.grid(alpha=0.3)
    for ax in axes[-1]:
        ax.set_xlabel("lead time (h)")
    axes[0, 0].legend()
    fig.suptitle(f"{cfg.name}: verified against {truth}")
    fig.tight_layout()


## Table at selected lead times

In [ ]:
rows = []
for truth, by_model in metrics.items():
    for model, m in by_model.items():
        for h in LEADS_FOR_TABLES:
            lt = np.timedelta64(h, "h")
            if lt not in m["lead_time"].values:
                continue
            v = m.sel(lead_time=lt)
            rows.append({"truth": truth, "model": model, "lead (h)": h,
                         "CRPS": float(v.crps), "SCRPS": float(v.scrps), "DSS": float(v.dss),
                         "RMSE": float(v.ens_mean_rmse), "spread": float(v.spread),
                         "SSR": float(v.spread_skill), "ES": float(v.energy_score)})
pd.DataFrame(rows).set_index(["truth", "lead (h)", "model"]).sort_index().round(3)


## Calibration: PIT histograms

Flat = calibrated. U-shaped = too little spread (the truth often falls outside the ensemble); dome-shaped = too much; sloped = biased.

In [ ]:
for truth, by_model in metrics.items():
    fig, axes = plt.subplots(1, len(LEADS_FOR_TABLES), figsize=(4.5 * len(LEADS_FOR_TABLES), 3.2), sharey=True)
    for ax, h in zip(np.atleast_1d(axes), LEADS_FOR_TABLES):
        for model, m in by_model.items():
            lt = np.timedelta64(h, "h")
            if lt in m["lead_time"].values:
                pit = m["pit_hist"].sel(lead_time=lt)
                ax.step(np.arange(pit.size + 1) / pit.size, np.r_[pit.values, pit.values[-1]],
                        where="post", label=model)
        ax.axhline(1, color="k", lw=0.8, ls="--"); ax.set_title(f"+{h} h"); ax.set_xlabel("PIT")
    np.atleast_1d(axes)[0].set_ylabel("density"); np.atleast_1d(axes)[0].legend()
    fig.suptitle(f"PIT vs {truth}"); fig.tight_layout()


## Spread-skill by spread bin

Points on the diagonal mean that when the ensemble says it is uncertain, it is. With 10 members the sample spread is noisy, which flattens this diagram even for a calibrated ensemble (see `scores.case_statistics`): compare models with each other rather than with the diagonal alone.

In [ ]:
for truth, by_model in metrics.items():
    fig, axes = plt.subplots(1, len(LEADS_FOR_TABLES), figsize=(4.5 * len(LEADS_FOR_TABLES), 3.8))
    for ax, h in zip(np.atleast_1d(axes), LEADS_FOR_TABLES):
        lt = np.timedelta64(h, "h")
        top = 0
        for model, m in by_model.items():
            if lt in m["lead_time"].values:
                sp, rm = m["binned_spread"].sel(lead_time=lt), m["binned_rmse"].sel(lead_time=lt)
                ok = np.isfinite(sp) & np.isfinite(rm)
                ax.plot(sp[ok], rm[ok], marker="o", label=model)
                top = max(top, float(np.nanmax([sp[ok].max(), rm[ok].max()])))
        ax.plot([0, top], [0, top], "k--", lw=0.8)
        ax.set_title(f"+{h} h"); ax.set_xlabel("spread (K)"); ax.set_ylabel("RMSE (K)")
    np.atleast_1d(axes)[0].legend(); fig.suptitle(f"binned spread-skill vs {truth}"); fig.tight_layout()


## Coming next

Climatological ensemble baseline (skill scores), extremes (Brier, twCRPS, occurrence ratio, tail calibration, record misses), block-bootstrap confidence intervals and Diebold–Mariano tests, and station verification.